# 01 · Quality flags review (step 2)

Shows how often each quality flag fires per decade, and random flagged regions next to their scanned clipping, raw OCR and post-corrected text. Judge whether each flag is right.

Needs `python -m hisrag.ingest` and `python -m hisrag.normalize text` to have run.

| Flag | Meaning |
|---|---|
| `pc_repetition` | post-correction looped or duplicated text (a 4-word sequence ≥4×, at least 2× as often as in the OCR) |
| `pc_expanded` | post-correction is >1.3× as long as the OCR (duplicated or added text) |
| `pc_unsupported` | <60 % of the corrected text's character trigrams occur in the OCR: reconstructed from noise, possibly invented |
| `too_short` | under 30 characters |
| `death_register` | an entry of the Vienna death register (name, occupation, age, address, cause of death), not an ad |

In [ ]:
import html
from IPython.display import HTML, display
from hisrag.data import query

FLAGS = ["pc_repetition", "pc_expanded", "pc_unsupported", "too_short", "death_register"]
query("SELECT decade, count(*) AS regions, "
      + ", ".join(f"sum(flag_{f}::int) AS {f}" for f in FLAGS)
      + ", round(100 * avg((n_flags > 0)::int), 1) AS pct_any FROM ad_text JOIN ads USING (ad_id) GROUP BY 1 ORDER BY 1")

In [ ]:
def show(where: str, n: int = 5, seed: int = 0):
    """Random regions matching a SQL condition on ads/ad_text, with clipping, OCR and correction side by side."""
    rows = query(f"SELECT t.*, a.label, a.text, a.text_ocr, a.iiif_link FROM ad_text t JOIN ads a USING (ad_id) WHERE {where} "
                 f"ORDER BY hash(ad_id || '{seed}') LIMIT {n}")
    for r in rows.itertuples():
        flags = [f for f in FLAGS if getattr(r, f"flag_{f}")]
        esc = lambda s: html.escape(s or "").replace("\n", "<br>")
        display(HTML(f"""
        <div style="border-top:2px solid #888;margin-top:14px;padding-top:6px">
          <b>{r.ad_id}</b> · {r.year} · {r.label} · flags: <b>{", ".join(flags) or "none"}</b>
          · support {r.ocr_support:.2f} · length ratio {r.len_ratio:.2f} · max repeat {r.repeat_max}
          <div><img src="{r.iiif_link}" style="max-width:100%;max-height:420px;margin:6px 0"></div>
          <div style="display:flex;gap:16px;font-size:12px">
            <div style="flex:1"><i>OCR</i><br>{esc(r.text_ocr)}</div>
            <div style="flex:1"><i>post-corrected</i><br>{esc(r.text)}</div>
          </div>
        </div>"""))

## `pc_unsupported`

In [ ]:
show("flag_pc_unsupported")

## `pc_repetition`

In [ ]:
show("flag_pc_repetition")

## `pc_expanded`

In [ ]:
show("flag_pc_expanded")

## `death_register`

In [ ]:
show("flag_death_register")

## `too_short`

In [ ]:
show("flag_too_short")

## Unflagged, for comparison
Change `seed` to see other samples.

In [ ]:
show("n_flags = 0", seed=1)